In [149]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated , Literal
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage , SystemMessage
from dotenv import load_dotenv
from pydantic import Field , BaseModel
import operator


In [139]:
load_dotenv()

True

In [140]:
llm = HuggingFaceEndpoint(repo_id='Qwen/Qwen2.5-7B-Instruct')

In [141]:
class EvaluationSchema(BaseModel):
    
    evaluation : Literal["approved","needs_improvement"] = Field(...,description="final evaluation result. ")
    feedback : str = Field(..., description= "Consstructive feedback for the tweet.")
    

In [142]:
generator_llm = ChatHuggingFace(llm=llm)
evaluator_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
optimizer_llm = ChatHuggingFace(llm=llm)
structured_evaluator_llm = evaluator_llm.with_structured_output(EvaluationSchema)

In [150]:
#state 
class tweetstate(TypedDict):
    
    topic : str
    tweet : str
    evaluation : Literal['approved','needs_improvement']
    feedback : str
    iteration : int
    max_iteration : int 
    
    
    tweet_history : Annotated[list[str],operator.add]
    feedback_history : Annotated[list[str],operator.add]

In [152]:
def generate_tweet(state:tweetstate):
    
    messages = [
        SystemMessage(content='You are a funny and clever twitter/X influencer.'),
        HumanMessage(content=f'''
    write a short , original and hilarious tweet on the topic "{state['topic']}".
    
    rules:
    -Do not use question answer format.
    -use 280 character.
    -use observation humor , irony, sarcasm or cultural refrences.
    -Think in name logic , puchlines , or relatable takes.
    -Use simple , day to day english
    ''')
    ]
    
    tweet = generator_llm.invoke(messages).content
    
    return {'tweet':tweet,'tweet_history':[tweet]}
    
    
    
def evaluate_tweet(state:tweetstate):
    
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
    Evaluate the following tweet:
        
    Tweet: "{state['tweet']}"
    
    Use the criteria below to evaluate the tweet:
        
    1. Originality Is this fresh, or have you seen it a hundred times before?
    2. Humor Did it genuinely make you smile, laugh, or chuckle?
    3. Punchiness Is it short, sharp, and scroll-stopping?
    4. Virality Potential Would people retweet or share it?
    5. Format Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?
    
    Auto-reject if:
    -It's written in question-answer format (e.g., "Why did..." or "What happens when...")
    -It exceeds 280 characters
    -It reads like a traditional setup-punchline joke
    -Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-uncle universe" or vague summaries)
    
    ### Respond ONLY in structured format:
    -evaluation: "approved" or "needs_improvement"
    -feedback: One paragraph explaining the strengths and weaknesses
    """)
    ]
    
    response = structured_evaluator_llm.invoke(messages)
    
    return {'evaluation':response.evaluation,'feedback':response.feedback,'feedback_history':[response.feedback]}

def optimize_tweet(state:tweetstate):
    
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
    Improve the tweet based on this feedback:
    "{state['feedback']}"
    
    Topic: "{state['topic']}"
    Original Tweet:
    {state['tweet']}
    Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
    """)
    ]
    
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1
    
    return {'tweet':response,'iteration':iteration,'tweet_history':[response]}
    
    

In [145]:
def route_evaluation(state:tweetstate):
    
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'


    

In [146]:
graph = StateGraph(tweetstate)

#add nodes
graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)

graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')

graph.add_conditional_edges('evaluate',route_evaluation,{'approved':END,'needs_improvement':'optimize'})

graph.add_edge('optimize','evaluate')

workflow = graph.compile()

In [153]:
initial_state = {
    "topic": "Indian Railways",
    "iteration":1,
    "max_iteration":5
}

workflow.invoke(initial_state)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 20.19867959s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMo